# Lektion 18: Parallelität, Subgraphs & Supervisor



| | |
|---|---|
| **Zeit** | ~75 Minuten |
| **Schwierigkeit** | Fortgeschritten |
| **Voraussetzungen** | Lektion 13–17 |
| **Du baust** | Drei Orchestrierungs-Muster: parallele Prüfung mit Merge, ein Graph im Graph (Subgraph) und einen Supervisor |

## 🎯 Nach dieser Lektion kannst du:

- Nodes **parallel** ausführen und Ergebnisse an einer Barriere zusammenführen

- Schreibkonflikte vermeiden (jedes Feld gehört einem Zweig)

- einen kompletten Graphen als Subgraph in einen größeren einbauen (Kapselung beweisen)

- einen Supervisor mit **Runden-Limit** bauen (keine Endlos-Delegation)

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Drei Muster für große Systeme

1. **Parallelität** – **Metapher: die Kommission.** Drei Prüfer erhalten dieselbe Akte, arbeiten gleichzeitig, dann sitzen alle zusammen (Merge-Barriere).
2. **Subgraph** – **Metapher: die Abteilung.** Der Chef-Graph kennt nur die Abteilung als Black Box; wie sie intern arbeitet, ist ihre Sache.
3. **Supervisor** – **Metapher: die Projektleitung.** Sie delegiert an Spezialisten, sammelt Berichte und entscheidet: weiter delegieren oder FINISH. **Spezialisten reden nie direkt miteinander** (sonst explodieren die Kommunikationspfade: n·(n−1)/2).

## Muster 1: Parallelität mit Merge

In [2]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

class KommissionsZustand(TypedDict):
    text: str
    woerter: int        # Zweig 1 schreibt NUR dieses Feld
    zeichen: int        # Zweig 2 schreibt NUR dieses Feld
    laenge: float       # Zweig 3 schreibt NUR dieses Feld
    bericht: str

g = StateGraph(KommissionsZustand)
g.add_node("zaehle_woerter", lambda s: {"woerter": len(s["text"].split())})
g.add_node("zaehle_zeichen", lambda s: {"zeichen": len(s["text"])})
g.add_node("wortlaenge", lambda s: {"laenge": round(
    sum(len(w) for w in s["text"].split()) / max(1, len(s["text"].split())), 2)})
g.add_node("zusammenfuehren", lambda s: {"bericht": (
    f"Kommissionsbericht: {s['woerter']} Woerter, {s['zeichen']} Zeichen, "
    f"durchschnittliche Wortlaenge {s['laenge']}.")})

for name in ("zaehle_woerter", "zaehle_zeichen", "wortlaenge"):
    g.add_edge(START, name)               # drei parallele Zweige...
    g.add_edge(name, "zusammenfuehren")    # ...die alle am Merge muessen liefern
g.add_edge("zusammenfuehren", END)
parallel_app = g.compile()

print(parallel_app.invoke({"text": "LangGraph macht paralleles Arbeiten moeglich",
                           "woerter": 0, "zeichen": 0, "laenge": 0.0, "bericht": ""})["bericht"])

Kommissionsbericht: 5 Woerter, 44 Zeichen, durchschnittliche Wortlaenge 8.0.


### 🔎 Was sehen wir hier?
**Die zwei Regeln:** (1) Jeder Zweig schreibt sein **eigenes** Feld – schreiben alle dasselbe, gewinnt zufällig einer (Last-Write-Wins = Datenverlust!). (2) Der Merge läuft erst, wenn **alle** Zweige geliefert haben – die Barriere synchronisiert automatisch.

## Muster 2: Der Subgraph (Abteilung)

In [3]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict

# ---- Abteilung: Qualitaets-Loop (eigenstaendiger Graph!) ----------------
class LoopZustand(TypedDict):
    text: str; bewertung: int; versuche: int; max_versuche: int

quali_loop = StateGraph(LoopZustand)
quali_loop.add_node("entwerfen", lambda s: {
    "text": f"{s['text']} [Politur {s['versuche'] + 1}]",
    "versuche": s["versuche"] + 1})
quali_loop.add_node("pruefen", lambda s: {"bewertung": min(100, 30 + 25 * (s["versuche"] + 1))})
quali_loop.add_edge(START, "entwerfen")
quali_loop.add_edge("entwerfen", "pruefen")
quali_loop.add_conditional_edges("pruefen", lambda s: "fertig"
    if s["bewertung"] >= 60 or s["versuche"] >= s["max_versuche"] else "nachbessern",
    {"fertig": END, "nachbessern": "entwerfen"})
qualitaets_abteilung = quali_loop.compile()

# Beweis 1: standalone laeuft die Abteilung:
solo = qualitaets_abteilung.invoke({"text": "Solo", "bewertung": 0, "versuche": 0, "max_versuche": 3})
print("Subgraph standalone:", solo["text"])

# ---- Das Gebaeude: Subgraph als EIN Node ---------------------------------
class ProjektZustand(TypedDict):
    auftrag: str; freigegebenes_produkt: str; status: str

def qualitaet(state: ProjektZustand) -> dict:
    """Adapter: einzige Nahtstelle zwischen aeusserem und innerem State."""
    ergebnis = qualitaets_abteilung.invoke({
        "text": state["auftrag"], "bewertung": 0,
        "versuche": 0, "max_versuche": 4})
    return {"freigegebenes_produkt": ergebnis["text"]}

gebaeude = StateGraph(ProjektZustand)
gebaeude.add_node("qualitaet", qualitaet)          # <- ganzer Graph als Station!
gebaeude.add_node("abschliessen", lambda s: {"status": "versandbereit"})
gebaeude.add_edge(START, "qualitaet")
gebaeude.add_edge("qualitaet", "abschliessen")
gebaeude.add_edge("abschliessen", END)
gebaeude_app = gebaeude.compile()

final = gebaeude_app.invoke({"auftrag": "Broschuere v1", "freigegebenes_produkt": "", "status": ""})
print("Finaler aeusserer State:")
for feld, wert in final.items():
    print(f"  {feld}: {wert}")
print("-> 'bewertung'/'versuche' fehlen: Kapselung wirkt!")

Subgraph standalone: Solo [Politur 1]
Finaler aeusserer State:
  auftrag: Broschuere v1
  freigegebenes_produkt: Broschuere v1 [Politur 1]
  status: versandbereit
-> 'bewertung'/'versuche' fehlen: Kapselung wirkt!


## ✏️ Deine Aufgabe: Der Supervisor mit Runden-Limit

Du baust die Projektleitung: drei Stationen (Recherche → Schreiben → FINISH) mit Beweis gegen Endlos-Delegation.

**Anforderungen:**

- Supervisor-Node: Delegations-Reihenfolge per Regel (offline): keine Stichpunkte → `recherche_agent`; kein Bericht → `schreib_agent`; sonst `FINISH`.

- Runden-Limit: nach `max_runden` erzwingt der Supervisor FINISH (im Zustand `runden` mitzählen!).

- Ergebnis-Log im State protokolliert jede Delegation.

- Beweise: normaler Lauf endet mit FINISH nach Bericht; ein Lauf mit `max_runden=1` bricht kontrolliert ab.

**Schreibe deinen Code in die folgende Zelle:**

In [4]:
class ProjektZustand(TypedDict):
    auftrag: str
    naechster_schritt: str
    stichpunkte: list
    bericht: str
    ergebnis_log: list
    runden: int
    max_runden: int

def supervisor(state):
    if state["runden"] >= state["max_runden"]:
        schritt = "FINISH"
    elif not state["stichpunkte"]:
        schritt = "recherche_agent"
    elif not state["bericht"]:
        schritt = "schreib_agent"
    else:
        schritt = "FINISH"
    print(f"  Supervisor -> {schritt} (Runde {state['runden'] + 1})")
    return {"naechster_schritt": schritt, "runden": state["runden"] + 1,
            "ergebnis_log": state["ergebnis_log"] + [f"supervisor: {schritt}"]}

def recherche_agent(state):
    punkte = ["RAG kombiniert Suche und Generierung.",
              "Embeddings machen Aehnlichkeit messbar."]
    return {"stichpunkte": punkte,
            "ergebnis_log": state["ergebnis_log"] + ["recherche_agent: 2 Stichpunkte"]}

def schreib_agent(state):
    return {"bericht": f"Bericht zu '{state['auftrag']}': " + " ".join(state["stichpunkte"]),
            "ergebnis_log": state["ergebnis_log"] + ["schreib_agent: Bericht erstellt"]}

sg = StateGraph(ProjektZustand)
sg.add_node("supervisor", supervisor)
sg.add_node("recherche_agent", recherche_agent)
sg.add_node("schreib_agent", schreib_agent)
sg.add_edge(START, "supervisor")
sg.add_edge("recherche_agent", "supervisor")
sg.add_edge("schreib_agent", "supervisor")
sg.add_conditional_edges("supervisor", lambda s: s["naechster_schritt"],
    {"recherche_agent": "recherche_agent",
     "schreib_agent": "schreib_agent", "FINISH": END})
sup_app = sg.compile()

e = sup_app.invoke({"auftrag": "RAG", "naechster_schritt": "", "stichpunkte": [],
                    "bericht": "", "ergebnis_log": [], "runden": 0, "max_runden": 5})
print("Log:", *e["ergebnis_log"], sep="\n  ")

  Supervisor -> recherche_agent (Runde 1)
  Supervisor -> schreib_agent (Runde 2)
  Supervisor -> FINISH (Runde 3)
Log:
  supervisor: recherche_agent
  recherche_agent: 2 Stichpunkte
  supervisor: schreib_agent
  schreib_agent: Bericht erstellt
  supervisor: FINISH


**💡 Warum diese Loesung funktioniert:**

Der Supervisor ist selbst nur ein Node mit klarer Logik – aber seine **Reihenfolge-Regeln** (erst recherchieren, dann schreiben) machen das System robust. Das Runden-Limit ist der Kosten-Airbag: Ohne es könnte ein LLM-Supervisor (mit echtem Modell) endlos delegieren. Im Finale steckt ein kleiner Verwandter in `final-project/.../graphs/workflow.py`.

### ✅ Richtig, wenn:

- [ ] Der normale Lauf delegiert Recherche, dann Schreiben, dann FINISH.

- [ ] Mit `max_runden=1` terminiert der Lauf trotzdem (kein Endlos-Loop!).

- [ ] Das Log zeigt jede Delegation inklusive Runde.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [5]:
e = sup_app.invoke({"auftrag": "RAG", "stichpunkte": [], "bericht": "",
                    "ergebnis_log": [], "runden": 0, "max_runden": 5})
assert e["bericht"], "Der Bericht muss entstehen."
assert any("recherche" in x for x in e["ergebnis_log"])
knapp = sup_app.invoke({"auftrag": "RAG", "stichpunkte": [], "bericht": "",
                        "ergebnis_log": [], "runden": 0, "max_runden": 1})
assert knapp["runden"] <= 2, "Limit muss greifen."
print("Alle Prüfungen bestanden! ✅")

  Supervisor -> recherche_agent (Runde 1)
  Supervisor -> schreib_agent (Runde 2)
  Supervisor -> FINISH (Runde 3)
  Supervisor -> recherche_agent (Runde 1)
  Supervisor -> FINISH (Runde 2)
Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Der Supervisor gibt `{{"naechster_schritt": ..., "runden": s["runden"]+1, "ergebnis_log": ...}}` zurück – Achtung: du brauchst ein Feld `naechster_schritt` im State.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Die Weiche nach dem Supervisor: `lambda s: s["naechster_schritt"]` mit Mapping zu den Nodes und `END`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Agenten melden sich zurück beim Supervisor: `add_edge("recherche_agent", "supervisor")`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Was sind die zwei klassischen Supervisor-Fehler – und welche zwei Bremsen hast du verbaut?

2. Wann ist ein einfacher sequenzieller Graph besser als ein Supervisor-Team?

## ➡️ Weiter

**🏁 Lektion 19:** `notebooks/10_capstone/19_capstone_systemtest.ipynb` – du testest das komplette finale System (Frontend → Backend → Agent → beide Wissensquellen) von innen.